# 資料處理與分析概念

## 學習目標

完成本 Notebook 後，你將能夠：

1. 使用 Python 計算平均數、中位數、眾數等中央趨勢統計量。
2. 使用全距、四分位距、變異數與標準差理解資料分散程度。
3. 透過簡單圖表進行探索式資料分析，觀察分布、異常值與變數關係。
4. 理解虛無假設、對立假設、顯著水準與 p 值的意義。
5. 使用輕量資料集完成一個從 EDA 到假說檢定的小型分析流程。


In [ ]:
# ── 環境設定 ────────────────────────────────────
# 載入本章節所需的 Python 套件，並建立一份可重複使用的範例資料。

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from collections import Counter
from scipy import stats

np.random.seed(42)

scores = np.array([68, 72, 75, 78, 80, 82, 82, 85, 88, 90, 95, 100])
study_hours = np.array([1.5, 2.0, 2.2, 2.8, 3.0, 3.5, 3.8, 4.1, 4.5, 5.0, 5.5, 6.0])

sales_df = pd.DataFrame({
    '學習時數': study_hours,
    '測驗分數': scores,
    '是否完成練習': ['否', '否', '否', '是', '是', '是', '是', '是', '是', '是', '是', '是']
})

print('資料筆數:', len(sales_df))
print(sales_df.head())


## 核心概念說明

在 AI 與機器學習中，資料品質會直接影響模型品質。進入模型訓練之前，通常會先做資料處理與資料分析，了解資料是否完整、是否有異常值、分布是否合理，以及變數之間是否可能存在關係。

常見統計量可分成兩大類：

- 中央趨勢：描述資料集中位置，例如平均數、中位數、眾數。
- 分散程度：描述資料變動大小，例如全距、四分位距、變異數、標準差。

平均數容易受到極端值影響；中位數對極端值較穩健；眾數適合觀察最常出現的類別或數值。分散程度則能幫助我們判斷資料是否穩定，例如標準差越大，代表資料點離平均值通常越遠。


In [ ]:
# ── 示範：中央趨勢與極端值影響 ───────────────────────────
# 這段程式碼示範平均數、中位數、眾數的計算，並比較加入極端值後對統計量的影響。

import numpy as np
from collections import Counter

scores = np.array([68, 72, 75, 78, 80, 82, 82, 85, 88, 90, 95, 100])
scores_with_outlier = np.append(scores, 200)

def mode_value(values):
    counts = Counter(values)
    max_count = max(counts.values())
    return [value for value, count in counts.items() if count == max_count]

print('原始資料')
print('平均數:', np.mean(scores))
print('中位數:', np.median(scores))
print('眾數:', mode_value(scores))

print('\n加入極端值 200 後')
print('平均數:', np.mean(scores_with_outlier))
print('中位數:', np.median(scores_with_outlier))
print('眾數:', mode_value(scores_with_outlier))

print('\n觀察：平均數明顯上升，中位數變化較小，表示中位數較不容易受到極端值影響。')


## 分散度與探索式資料分析

分散度衡量資料的離散程度。常見指標包含：

- 全距：最大值減最小值，簡單但非常容易受到極端值影響。
- 四分位距：第三四分位數減第一四分位數，常用於觀察中間 50% 資料的變動範圍。
- 變異數與標準差：描述資料相對於平均數的平均偏離程度。

探索式資料分析強調先觀察資料，再形成問題與假設。常見方法包含摘要統計、直方圖、盒鬚圖與散佈圖。這些方法能協助找出分布型態、異常值與變數關係。


In [ ]:
# ── 示範：分散度與 EDA 圖表 ──────────────────────────
# 這段程式碼計算全距、四分位距、變異數與標準差，並使用直方圖、盒鬚圖和散佈圖觀察資料。

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

scores = np.array([68, 72, 75, 78, 80, 82, 82, 85, 88, 90, 95, 100])
study_hours = np.array([1.5, 2.0, 2.2, 2.8, 3.0, 3.5, 3.8, 4.1, 4.5, 5.0, 5.5, 6.0])

q1 = np.percentile(scores, 25)
q3 = np.percentile(scores, 75)
iqr = q3 - q1

print('全距:', np.max(scores) - np.min(scores))
print('第一四分位數 Q1:', q1)
print('第三四分位數 Q3:', q3)
print('四分位距 IQR:', iqr)
print('變異數:', np.var(scores, ddof=1))
print('標準差:', np.std(scores, ddof=1))

fig, axes = plt.subplots(1, 3, figsize=(15, 4))

axes[0].hist(scores, bins=6, edgecolor='black')
axes[0].set_title('測驗分數直方圖')
axes[0].set_xlabel('分數')
axes[0].set_ylabel('人數')

axes[1].boxplot(scores, vert=True)
axes[1].set_title('測驗分數盒鬚圖')
axes[1].set_ylabel('分數')

axes[2].scatter(study_hours, scores)
axes[2].set_title('學習時數與測驗分數')
axes[2].set_xlabel('學習時數')
axes[2].set_ylabel('測驗分數')

plt.tight_layout()
plt.show()


## 假說檢定與統計推論

當我們從樣本推論母體時，樣本統計量與母體參數之間可能存在抽樣變異，因此需要透過機率模型與假說檢定來量化不確定性。

假說檢定常見流程為：提出假設、蒐集資料、計算檢定統計量、取得 p 值、根據顯著水準做決策。

常用名詞：

- 虛無假設 H0：通常表示沒有差異、沒有效果或沒有關係。
- 對立假設 Ha：通常表示存在差異、存在效果或存在關係。
- 顯著水準 alpha：常見為 0.05，表示可接受的第一型錯誤風險門檻。
- p 值：在 H0 為真時，觀察到目前或更極端結果的機率。

判斷方式通常是：若 p 值小於 alpha，拒絕 H0；若 p 值大於或等於 alpha，則沒有足夠證據拒絕 H0。


In [ ]:
# ── 實際應用：從 EDA 到假說檢定 ────────────────────────
# 這段程式碼示範如何比較兩組學生的測驗分數，先做描述統計，再使用 t 檢定判斷平均分數是否有顯著差異。

import numpy as np
import pandas as pd
from scipy import stats

np.random.seed(7)

traditional = np.array([70, 72, 75, 76, 78, 80, 81, 83, 84, 85])
ai_assisted = np.array([76, 78, 79, 82, 84, 85, 87, 88, 90, 92])

summary = pd.DataFrame({
    '組別': ['傳統學習', 'AI 輔助學習'],
    '平均數': [np.mean(traditional), np.mean(ai_assisted)],
    '中位數': [np.median(traditional), np.median(ai_assisted)],
    '標準差': [np.std(traditional, ddof=1), np.std(ai_assisted, ddof=1)]
})

print(summary)

alpha = 0.05
t_stat, p_value = stats.ttest_ind(ai_assisted, traditional, equal_var=False)

print('\n虛無假設 H0：兩組平均分數沒有差異')
print('對立假設 Ha：兩組平均分數有差異')
print('t 統計量:', round(t_stat, 4))
print('p 值:', round(p_value, 4))

if p_value < alpha:
    print('結論：p 值小於 0.05，拒絕 H0，兩組平均分數可能存在顯著差異。')
else:
    print('結論：p 值不小於 0.05，沒有足夠證據拒絕 H0。')


In [ ]:
# ── 自我測驗 ────────────────────────────────────
# 請完成下方 TODO 填空，實作中央趨勢、分散度與簡單假說檢定的功能。

import numpy as np
from scipy import stats

# 題目資料：某班 10 位學生的測驗分數
scores = np.array([60, 65, 70, 70, 75, 80, 85, 90, 95, 100])

# TODO 1：計算平均數
mean_score = None

# TODO 2：計算中位數
median_score = None

# TODO 3：計算全距
score_range = None

# TODO 4：計算樣本標準差，請使用 ddof=1
sample_std = None

print('平均數:', mean_score)
print('中位數:', median_score)
print('全距:', score_range)
print('樣本標準差:', sample_std)

# Expected:
# 平均數: 79.0
# 中位數: 77.5
# 全距: 40
# 樣本標準差: 約 13.7

# 進階 TODO 5：檢定這組學生的平均分數是否顯著不同於 75
# 提示：使用 stats.ttest_1samp(scores, popmean=75)
t_stat, p_value = None, None

print('t 統計量:', t_stat)
print('p 值:', p_value)

# Expected:
# t 統計量與 p 值會是數值輸出
# 若 p 值 < 0.05，代表平均分數與 75 有顯著差異；否則沒有足夠證據認為有顯著差異。
